In [22]:
import requests
import pandas as pd

print('Libraries imported')
print(f'Pandas: {pd.__version__}')

Libraries imported
Pandas: 3.0.3


In [23]:
# paste your API key from openweathermap.org
API_KEY = "a90ece615d6beed52df4e2e676335810"

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai',
          'Hyderabad', 'Kolkata', 'Pune', 'Jaipur',
          'Ahmedabad', 'Lucknow']

print(f'Cities to fetch: {CITIES}')

Cities to fetch: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur', 'Ahmedabad', 'Lucknow']


In [24]:
# EXTRACT — fetch weather for each city

weather_records = []

for city in CITIES:
    params = {'q': city, 'appid': API_KEY, 'units': 'metric'}
    response = requests.get(BASE_URL, params=params, timeout=10)

    if response.status_code == 200:
        data = response.json()
        weather_records.append({
            'city':        data['name'],
            'temperature': data['main']['temp'],
            'feels_like':  data['main']['feels_like'],
            'humidity':    data['main']['humidity'],
            'pressure':    data['main']['pressure'],
            'wind_speed':  data['wind']['speed'],
            'condition':   data['weather'][0]['description'],
            'visibility':  data.get('visibility', 0) // 1000  # metres to km
        })
        print(f'{city} -> {data["main"]["temp"]}°C, {data["weather"][0]["description"]}')
    else:
        print(f'{city} -> FAILED ({response.status_code})')

print(f'\nFetched: {len(weather_records)}/{len(CITIES)} cities')

Mumbai -> 34.99°C, smoke
Delhi -> 39.05°C, haze
Bangalore -> 30.67°C, scattered clouds
Chennai -> 38.79°C, few clouds
Hyderabad -> 37.23°C, few clouds
Kolkata -> 32.97°C, haze
Pune -> 35.22°C, few clouds
Jaipur -> 42.62°C, haze
Ahmedabad -> 38.02°C, smoke
Lucknow -> 34.99°C, haze

Fetched: 10/10 cities


In [25]:
# fallback data — run this only if API key is not working

if len(weather_records) == 0:
    print('API not available, using fallback data')
    weather_records = [
        {'city': 'Mumbai',    'temperature': 32.5, 'feels_like': 36.0, 'humidity': 78, 'pressure': 1009, 'wind_speed': 5.2, 'condition': 'partly cloudy', 'visibility': 8},
        {'city': 'Delhi',     'temperature': 39.0, 'feels_like': 42.0, 'humidity': 20, 'pressure': 998,  'wind_speed': 4.1, 'condition': 'haze',          'visibility': 5},
        {'city': 'Bangalore', 'temperature': 26.0, 'feels_like': 27.0, 'humidity': 72, 'pressure': 1013, 'wind_speed': 3.5, 'condition': 'cloudy',        'visibility': 10},
        {'city': 'Chennai',   'temperature': 33.0, 'feels_like': 37.0, 'humidity': 68, 'pressure': 1008, 'wind_speed': 4.8, 'condition': 'haze',          'visibility': 6},
        {'city': 'Hyderabad', 'temperature': 30.5, 'feels_like': 33.0, 'humidity': 55, 'pressure': 1010, 'wind_speed': 3.9, 'condition': 'haze',          'visibility': 7},
        {'city': 'Kolkata',   'temperature': 27.0, 'feels_like': 30.0, 'humidity': 85, 'pressure': 1007, 'wind_speed': 2.8, 'condition': 'thunderstorm',  'visibility': 4},
        {'city': 'Pune',      'temperature': 29.3, 'feels_like': 31.0, 'humidity': 55, 'pressure': 1014, 'wind_speed': 3.1, 'condition': 'partly cloudy', 'visibility': 9},
        {'city': 'Jaipur',    'temperature': 40.1, 'feels_like': 43.0, 'humidity': 22, 'pressure': 998,  'wind_speed': 5.5, 'condition': 'sunny',         'visibility': 12},
        {'city': 'Ahmedabad', 'temperature': 33.0, 'feels_like': 36.0, 'humidity': 55, 'pressure': 1005, 'wind_speed': 4.0, 'condition': 'smoke',         'visibility': 6},
        {'city': 'Lucknow',   'temperature': 34.0, 'feels_like': 37.0, 'humidity': 46, 'pressure': 1002, 'wind_speed': 3.7, 'condition': 'haze',          'visibility': 5},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live data for {len(weather_records)} cities')

Using live data for 10 cities


In [26]:
# create raw dataframe
df_raw = pd.DataFrame(weather_records)

print(f'Shape: {df_raw.shape}')
print(df_raw)

Shape: (10, 8)
        city  temperature  feels_like  humidity  pressure  wind_speed  \
0     Mumbai        34.99       41.99        55      1009        6.69   
1      Delhi        39.05       42.14        32      1003        5.66   
2  Bengaluru        30.67       33.48        57      1012        5.36   
3    Chennai        38.79       45.79        51      1006        4.63   
4  Hyderabad        37.23       40.14        36      1007        6.17   
5    Kolkata        32.97       39.97        66      1004        5.14   
6       Pune        35.22       34.58        28      1009        4.18   
7     Jaipur        42.62       40.23        12      1001        6.69   
8  Ahmedabad        38.02       38.83        28      1005        7.72   
9    Lucknow        34.99       40.27        49      1001        5.14   

          condition  visibility  
0             smoke           6  
1              haze           4  
2  scattered clouds           8  
3        few clouds           6  
4        fe

In [27]:
# check data quality
print('Missing values:')
print(df_raw.isnull().sum())
print(f'\nDuplicates: {df_raw.duplicated().sum()}')
print(f'\nData types:')
print(df_raw.dtypes)

Missing values:
city           0
temperature    0
feels_like     0
humidity       0
pressure       0
wind_speed     0
condition      0
visibility     0
dtype: int64

Duplicates: 0

Data types:
city               str
temperature    float64
feels_like     float64
humidity         int64
pressure         int64
wind_speed     float64
condition          str
visibility       int64
dtype: object


In [28]:
# TRANSFORM — clean and add new columns
df = df_raw.copy()

# fix casing
df['city']      = df['city'].str.strip().str.title()
df['condition'] = df['condition'].str.strip().str.title()

# round numbers
df['temperature'] = df['temperature'].round(1)
df['feels_like']  = df['feels_like'].round(1)

# heat category
def heat_category(temp):
    if temp >= 40:   return 'Extreme Heat'
    elif temp >= 35: return 'Very Hot'
    elif temp >= 30: return 'Hot'
    elif temp >= 25: return 'Warm'
    else:            return 'Comfortable'

df['heat_category'] = df['temperature'].apply(heat_category)

# humidity level
def humidity_level(h):
    if h >= 70:   return 'High'
    elif h >= 40: return 'Moderate'
    else:         return 'Low'

df['humidity_level'] = df['humidity'].apply(humidity_level)

# how much hotter it feels vs actual temp
df['feels_diff'] = (df['feels_like'] - df['temperature']).round(1)

print('Transform done')
print(df[['city', 'temperature', 'heat_category', 'humidity', 'humidity_level', 'condition']])

Transform done
        city  temperature heat_category  humidity humidity_level  \
0     Mumbai         35.0      Very Hot        55       Moderate   
1      Delhi         39.0      Very Hot        32            Low   
2  Bengaluru         30.7           Hot        57       Moderate   
3    Chennai         38.8      Very Hot        51       Moderate   
4  Hyderabad         37.2      Very Hot        36            Low   
5    Kolkata         33.0           Hot        66       Moderate   
6       Pune         35.2      Very Hot        28            Low   
7     Jaipur         42.6  Extreme Heat        12            Low   
8  Ahmedabad         38.0      Very Hot        28            Low   
9    Lucknow         35.0      Very Hot        49       Moderate   

          condition  
0             Smoke  
1              Haze  
2  Scattered Clouds  
3        Few Clouds  
4        Few Clouds  
5              Haze  
6        Few Clouds  
7              Haze  
8             Smoke  
9              H

In [29]:
# analysis
print('=== Weather Analysis ===')

hottest   = df.loc[df['temperature'].idxmax()]
coolest   = df.loc[df['temperature'].idxmin()]
most_humid = df.loc[df['humidity'].idxmax()]

print(f'Hottest city    : {hottest["city"]} ({hottest["temperature"]}°C)')
print(f'Coolest city    : {coolest["city"]} ({coolest["temperature"]}°C)')
print(f'Most humid city : {most_humid["city"]} ({most_humid["humidity"]}%)')

print(f'\nAvg temperature : {df["temperature"].mean():.1f}°C')
print(f'Avg humidity    : {df["humidity"].mean():.1f}%')
print(f'Avg wind speed  : {df["wind_speed"].mean():.1f} m/s')

print('\nHeat categories:')
print(df['heat_category'].value_counts())

print('\nHumidity levels:')
print(df['humidity_level'].value_counts())

=== Weather Analysis ===
Hottest city    : Jaipur (42.6°C)
Coolest city    : Bengaluru (30.7°C)
Most humid city : Kolkata (66%)

Avg temperature : 36.5°C
Avg humidity    : 41.4%
Avg wind speed  : 5.7 m/s

Heat categories:
heat_category
Very Hot        7
Hot             2
Extreme Heat    1
Name: count, dtype: int64

Humidity levels:
humidity_level
Moderate    5
Low         5
Name: count, dtype: int64


In [30]:
# validation
print(f'Rows          : {len(df)}')
print(f'Columns       : {len(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates    : {df.duplicated().sum()}')
print(f'Clean         : {df.isnull().sum().sum() == 0 and df.duplicated().sum() == 0}')

Rows          : 10
Columns       : 11
Missing values: 0
Duplicates    : 0
Clean         : True


In [31]:
# LOAD — save to CSV
df.to_csv('weather_data.csv', index=False)

print('Saved to weather_data.csv')
print(f'{len(df)} rows, {len(df.columns)} columns')
print('\nETL Pipeline Complete!')
print('EXTRACT   -> fetched from OpenWeatherMap API')
print('TRANSFORM -> cleaned and added new columns')
print('LOAD      -> saved to weather_data.csv')

Saved to weather_data.csv
10 rows, 11 columns

ETL Pipeline Complete!
EXTRACT   -> fetched from OpenWeatherMap API
TRANSFORM -> cleaned and added new columns
LOAD      -> saved to weather_data.csv
